# 02 Stadtreferenzmodell

## Zweck

Erstellen Sie den stabilen Stadtreferenzdatensatz, der als zentraler Verbindungsanker für EEA-, Wikipedia-, Open-Meteo-, Kafka-, Spark-, Gold-Tabellen und die endgültige Analyse verwendet wird.

## Eingaben

Lokale Konstanten für acht ausgewählte europäische Städte. In diesem Notebook wird keine externe Quelle aufgerufen.

## Ausgaben

- `data/silver/city_reference.csv`
- `data/silver/city_reference.parquet`

## Verwendete Technologien

Python, Pandas, pyarrow, Jupyter-Notizbuch.

## Konfiguration

Ausgabepfade sind projektbezogen und verwenden standardmäßig `DATA_DIR=data`.

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd

_env_root = os.getenv("PROJECT_ROOT")
if _env_root:
    PROJECT_ROOT = Path(_env_root).resolve()
elif Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = PROJECT_ROOT / Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))

print({"project_root": str(PROJECT_ROOT), "data_dir": str(DATA_DIR)})


## Implementierung

Die Städteliste ist bewusst klein und überschaubar. `city_id` ist der einzige Downstream-City-Join-Schlüssel; Freitext-Städtenamen sind nur Anzeigefelder.

### Definieren Sie das Stadtmodell

Das Projekt nutzt bewusst eine kleine, überprüfbare Gruppe von acht Städten. Jeder Datensatz enthält den stabilen Join-Schlüssel `city_id` sowie Koordinaten und Quellnotizen, die von späteren Notebooks verwendet werden.

In [ ]:
from pathlib import Path
import pandas as pd

SILVER_DIR = DATA_DIR / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

CITY_RECORDS = [
    {"city_id": "vienna_at", "city_name": "Vienna", "city_name_normalized": "vienna", "country_code": "AT", "latitude": 48.2082, "longitude": 16.3738, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Pilot aus Phase 1; alle Quellen sind mit Einschränkungen nutzbar.", "eea_station_selection_notes": "EEA-Stadtzuordnung wird in Notebook 03 über den jeweiligen API-Stadtnamen geprüft.", "wikipedia_url": "https://en.wikipedia.org/wiki/Vienna"},
    {"city_id": "berlin_de", "city_name": "Berlin", "city_name_normalized": "berlin", "country_code": "DE", "latitude": 52.5200, "longitude": 13.4050, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Pilot aus Phase 1; alle Quellen sind mit Einschränkungen nutzbar.", "eea_station_selection_notes": "EEA-Stadtzuordnung wird in Notebook 03 über den jeweiligen API-Stadtnamen geprüft.", "wikipedia_url": "https://en.wikipedia.org/wiki/Berlin"},
    {"city_id": "paris_fr", "city_name": "Paris", "city_name_normalized": "paris", "country_code": "FR", "latitude": 48.8566, "longitude": 2.3522, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Große europäische Hauptstadt für geografische Streuung.", "eea_station_selection_notes": "EEA-Stadtzuordnung wird in Notebook 03 über den jeweiligen API-Stadtnamen geprüft.", "wikipedia_url": "https://en.wikipedia.org/wiki/Paris"},
    {"city_id": "madrid_es", "city_name": "Madrid", "city_name_normalized": "madrid", "country_code": "ES", "latitude": 40.4168, "longitude": -3.7038, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Südeuropäische Vergleichsstadt.", "eea_station_selection_notes": "EEA-Stadtzuordnung wird in Notebook 03 über den jeweiligen API-Stadtnamen geprüft.", "wikipedia_url": "https://en.wikipedia.org/wiki/Madrid"},
    {"city_id": "rome_it", "city_name": "Rome", "city_name_normalized": "rome", "country_code": "IT", "latitude": 41.9028, "longitude": 12.4964, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Mediterrane Vergleichsstadt.", "eea_station_selection_notes": "EEA-Stadtzuordnung wird in Notebook 03 über den jeweiligen API-Stadtnamen geprüft.", "wikipedia_url": "https://en.wikipedia.org/wiki/Rome"},
    {"city_id": "amsterdam_nl", "city_name": "Amsterdam", "city_name_normalized": "amsterdam", "country_code": "NL", "latitude": 52.3676, "longitude": 4.9041, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Kompakter Stadtkontext und erwartete Abdeckung durch Messstationen.", "eea_station_selection_notes": "EEA-Stadtzuordnung wird in Notebook 03 über den jeweiligen API-Stadtnamen geprüft.", "wikipedia_url": "https://en.wikipedia.org/wiki/Amsterdam"},
    {"city_id": "warsaw_pl", "city_name": "Warsaw", "city_name_normalized": "warsaw", "country_code": "PL", "latitude": 52.2297, "longitude": 21.0122, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Mittel-/osteuropäische Vergleichsstadt.", "eea_station_selection_notes": "EEA-Stadtzuordnung wird in Notebook 03 über den jeweiligen API-Stadtnamen geprüft.", "wikipedia_url": "https://en.wikipedia.org/wiki/Warsaw"},
    {"city_id": "prague_cz", "city_name": "Prague", "city_name_normalized": "prague", "country_code": "CZ", "latitude": 50.0755, "longitude": 14.4378, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Mitteleuropäische Vergleichsstadt.", "eea_station_selection_notes": "EEA-Stadtzuordnung wird in Notebook 03 über den jeweiligen API-Stadtnamen geprüft.", "wikipedia_url": "https://en.wikipedia.org/wiki/Prague"},
]

REQUIRED_COLUMNS = ["city_id", "city_name", "city_name_normalized", "country_code", "latitude", "longitude"]

print(f"OK: {len(CITY_RECORDS)} Städte konfiguriert — SILVER_DIR: {SILVER_DIR}")


### Validierung und Materialisierung des Stadtmodells

Der Builder überprüft die `city_id`-Konvention, Eindeutigkeit, erforderliche Felder, Ländercodes und Koordinatenbereiche, bevor er einen DataFrame zurückgibt.

In [ ]:
def build_city_reference() -> pd.DataFrame:
    df = pd.DataFrame(CITY_RECORDS)
    expected_city_ids = df["city_name_normalized"] + "_" + df["country_code"].str.lower()
    if not (df["city_id"] == expected_city_ids).all():
        raise ValueError("city_id muss dem Muster <city_name_normalized>_<country_code> folgen")
    if not df["city_id"].is_unique:
        raise ValueError("city_id-Werte müssen eindeutig sein")
    if df[REQUIRED_COLUMNS].isna().any().any():
        raise ValueError("Erforderliche Felder der Städtereferenz dürfen keine Nullwerte enthalten")
    if not df["country_code"].str.fullmatch(r"[A-Z]{2}").all():
        raise ValueError("country_code muss aus zwei Großbuchstaben bestehen")
    if not df["latitude"].between(-90, 90).all():
        raise ValueError("Ungültiger Breitengrad")
    if not df["longitude"].between(-180, 180).all():
        raise ValueError("Ungültiger Längengrad")
    return df

city_reference_df = build_city_reference()
city_reference_df


## Validierung / Qualitätsprüfungen

Validieren Sie die Eindeutigkeit, die erforderlichen Felder, das Ländercodeformat, die Koordinatenbereiche und das Zurücklesen der Ausgabe.

### Überprüfen Sie die Stadtreferenz-Invarianten erneut

Diese Behauptungen machen die wichtigsten Annahmen in der Notebook-Ausgabe sichtbar, bevor Dateien beibehalten werden.

In [ ]:
assert len(city_reference_df) == 8, \
    f"8 Städte erwartet, erhalten {len(city_reference_df)}"
assert city_reference_df["city_id"].is_unique, \
    f"Doppelte city_id-Werte gefunden: {city_reference_df[city_reference_df['city_id'].duplicated()]['city_id'].tolist()}"
assert city_reference_df[REQUIRED_COLUMNS].notna().all().all(), \
    f"Nullwerte in erforderlichen Spalten: {city_reference_df[REQUIRED_COLUMNS].isna().sum()[lambda s: s > 0].to_dict()}"
assert city_reference_df["latitude"].between(-90, 90).all(), \
    f"Breitengrad außerhalb des gültigen Bereichs: {city_reference_df.loc[~city_reference_df['latitude'].between(-90, 90), ['city_id', 'latitude']]}"
assert city_reference_df["longitude"].between(-180, 180).all(), \
    f"Längengrad außerhalb des gültigen Bereichs: {city_reference_df.loc[~city_reference_df['longitude'].between(-180, 180), ['city_id', 'longitude']]}"

print(f"OK: alle 5 Stadtreferenz-Invarianten bestanden ({len(city_reference_df)} Städte, alle Koordinaten im gültigen Bereich)")


### Schreiben Sie die Stadtreferenz auf und lesen Sie sie noch einmal vor

Die Datensätze werden in einen DataFrame konvertiert und als CSV und Parquet gespeichert. CSV ist praktisch für die Inspektion; Parkett ist das Downstream-Pipeline-Format.

In [ ]:
csv_path = SILVER_DIR / "city_reference.csv"
parquet_path = SILVER_DIR / "city_reference.parquet"
city_reference_df.to_csv(csv_path, index=False)
city_reference_df.to_parquet(parquet_path, index=False)

roundtrip = pd.read_parquet(parquet_path)
assert len(roundtrip) == len(city_reference_df), \
    f"Abweichende Zeilenanzahl beim Parquet-Roundtrip: geschrieben: {len(city_reference_df)}, erneut gelesen: {len(roundtrip)}"
assert set(REQUIRED_COLUMNS).issubset(roundtrip.columns), \
    f"In Parquet fehlen erforderliche Spalten: {set(REQUIRED_COLUMNS) - set(roundtrip.columns)}"
roundtrip.head()


## Ergebnisse

Phase 2 erzeugt eine stabile Stadtreferenz CSV und eine Parquet-Datei für spätere Notebooks.

## Einschränkungen

Bei den Koordinaten handelt es sich um Näherungswerte für das Stadtzentrum. Bevölkerung, Fläche und Dichte sind kontextbezogene Felder und werden in Phase 4 aus Wikipedia befüllt, sofern sie analysierbar sind.

## Nächster Schritt

Führen Sie das Notebook `03_eea_batch_ingestion.ipynb` aus, um die Datei/den Stapel EEA-Quelle zu verarbeiten.